# Layout Model Comparison

Compare the current classical detector with optional document-layout backends. Optional backends skip gracefully when dependencies or local model paths are missing. No model downloads are performed by this notebook.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from utils.image.layout.strategies import (
    ClassicalLayoutStrategy,
    DocLayoutYOLOStrategy,
    GenericYOLOLayoutStrategy,
    LayoutParserStrategy,
    PaddleLayoutStrategy,
)

REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "data" / "notebook_outputs" / "layout_model_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional local model paths. Leave unset to skip these backends.
DOCLAYOUT_YOLO_MODEL = None
GENERIC_YOLO_MODEL = None

## Evaluation Samples

Use DFD pages with known detector behavior and Corpus 2 pages that showed generalization failures. Missing private/local files are skipped.

In [ ]:
SAMPLES = [
    {
        "id": "dfd_3335",
        "path": "data/corpora/donn_draeger/dfd_notes_master/original/original_img_3335.jpg",
        "notes": "large recovered figure",
    },
    {
        "id": "dfd_3344",
        "path": "data/corpora/donn_draeger/dfd_notes_master/original/original_img_3344.jpg",
        "notes": "labeled diagram",
    },
    {
        "id": "dfd_3397",
        "path": "data/corpora/donn_draeger/dfd_notes_master/original/original_img_3397.jpg",
        "notes": "sparse symbols and arrows",
    },
    {
        "id": "dfd_3352",
        "path": "data/corpora/donn_draeger/dfd_notes_master/original/original_img_3352.jpg",
        "notes": "broad label-heavy crop risk",
    },
    {
        "id": "dfd_3330",
        "path": "data/corpora/donn_draeger/dfd_notes_master/original/original_img_3330.jpg",
        "notes": "tall narrow visual strip risk",
    },
    # Add Corpus 2 sample paths from data/corpora/ad_hoc/corpus2/manifests/manifest.local.json.
    # {"id": "corpus2_example", "path": "data/corpora/ad_hoc/corpus2/original/example.jpg", "notes": "plain-text false positive case"},
]

def load_image(sample):
    path = REPO_ROOT / sample["path"]
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        print(f"Skipping missing sample: {sample['id']} -> {path}")
        return None
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

## Strategies

`classical_opencv` is the baseline. The optional strategies report skipped results unless their local dependencies and model configuration are available.

In [ ]:
strategies = [
    ClassicalLayoutStrategy(),
    PaddleLayoutStrategy(),
    LayoutParserStrategy(),
    DocLayoutYOLOStrategy({"model_path": DOCLAYOUT_YOLO_MODEL} if DOCLAYOUT_YOLO_MODEL else {}),
    GenericYOLOLayoutStrategy({"model_path": GENERIC_YOLO_MODEL} if GENERIC_YOLO_MODEL else {}),
]

In [ ]:
def draw_regions(image, regions, title):
    canvas = image.copy()
    for region in regions:
        color = (40, 180, 40)
        if region.region_type == "text":
            color = (220, 80, 80)
        elif region.region_type in {"figure", "diagram", "image", "photo"}:
            color = (40, 120, 240)
        cv2.rectangle(canvas, (region.x, region.y), (region.x2, region.y2), color, 3)
        label = region.region_type or "unknown"
        cv2.putText(canvas, label, (region.x, max(20, region.y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    plt.figure(figsize=(10, 12))
    plt.imshow(canvas)
    plt.title(title)
    plt.axis("off")
    plt.show()
    return canvas

def run_strategy(strategy, image):
    try:
        return strategy.detect(image)
    except Exception as exc:
        from utils.image.layout.strategy import skipped_result

        return skipped_result(strategy.name, f"error: {exc.__class__.__name__}: {exc}")

## Run Comparison

In [ ]:
summary_rows = []

for sample in SAMPLES:
    image = load_image(sample)
    if image is None:
        continue

    print(f"\n=== {sample['id']} ===")
    print(sample.get("notes", ""))
    sample_dir = OUTPUT_DIR / sample["id"]
    sample_dir.mkdir(parents=True, exist_ok=True)

    for strategy in strategies:
        result = run_strategy(strategy, image)
        row = {
            "sample_id": sample["id"],
            "strategy": result.strategy_name,
            "available": result.available,
            "skipped_reason": result.skipped_reason,
            "region_count": len(result.regions),
        }
        summary_rows.append(row)
        print(row)
        if not result.available:
            continue
        canvas = draw_regions(image, result.regions, f"{sample['id']} - {result.strategy_name}")
        out_path = sample_dir / f"{result.strategy_name}.png"
        cv2.imwrite(str(out_path), cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))

In [ ]:
summary_rows

## Review Notes

Record whether optional document-layout models improve text/figure separation over the baseline classical detector. If a backend performs well, test it in review mode only before considering any runtime integration.